In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, root_mean_squared_error
import category_encoders as ce
import xgboost as xgb
import lightgbm as lgb

In [2]:
# ==========================================
# 1. 데이터 로드
# ==========================================
print("데이터를 불러오는 중입니다...")
features_df = pd.read_csv('/Users/kwagminseo/Documents/SAS/data/train_features_merged.csv')
info_df = pd.read_csv("/Users/kwagminseo/Documents/SAS/data/train/train_customer_info.csv")
targets_df = pd.read_csv("/Users/kwagminseo/Documents/SAS/data/train_targets.csv")

데이터를 불러오는 중입니다...


In [3]:
# 원본에서 날아갔던 핵심 범주형 변수들만 선택해서 다시 가져옵니다.
info_cols = ['customer_id', 'gender', 'region_code', 'is_married', 'prefer_category', 'income_group']
df = pd.merge(features_df, info_df[info_cols], on='customer_id', how='left')

# 타겟(정답) 데이터도 병합합니다.
df = pd.merge(df, targets_df[['customer_id', 'target_churn', 'target_ltv', 'log_target_ltv']], on='customer_id', how='left')

In [4]:
# ==========================================
# 2. 파생 변수 생성 (🚨문제 변수 삭제 & 안전한 변수만 유지)
# ==========================================
df['age_group'] = pd.cut(df['age'], bins=[0, 29, 39, 49, 59, 100], labels=['20대이하', '30대', '40대', '50대', '60대이상'])
df['life_stage'] = df['age_group'].astype(str) + '_' + np.where(df['is_married'] == 1, '기혼', '미혼')
df['tenure_group'] = pd.qcut(df['tenure_days'], q=4, labels=['Q1(신규)', 'Q2(일반)', 'Q3(고참)', 'Q4(고인물)'])

# [수정 1] 과대 예측의 주범이었던 expected_baseline_ltv 삭제하고, 현실적인 순자산만 남김
df['net_worth'] = df['total_deposit_balance'] - df['total_loan_balance']
df['recency_ratio'] = df['recency_days'] / (df['tenure_days'] / (df['trans_count'] + 1e-6))

# 결측치/무한대 안전 처리
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(0, inplace=True)

# 금액 변수 로그 스케일링
skewed_cols = ['total_deposit_balance', 'total_loan_balance', 'total_spending', 'net_worth']
for col in skewed_cols:
    df[f'log_{col}'] = np.log1p(np.clip(df[col], 0, None))

In [7]:
# ③ 범주형 변수 타입 변경 (LightGBM 등 트리 모델용)
# 이렇게 category 타입으로 바꿔주면 One-Hot Encoding 없이도 모델이 스스로 범주형임을 인식합니다.
cat_cols = ['gender', 'region_code', 'is_married', 'prefer_category', 'income_group', 'age_group', 'life_stage', 'tenure_group']
for col in cat_cols:
    df[col] = df[col].astype('category')

In [8]:
# ==========================================
# 2. 검증셋 분할 (Train / Validation Split)
# ==========================================
# 타겟 변수와 ID 제외한 피처 설정
drop_cols = ['customer_id', 'target_churn', 'target_ltv', 'log_target_ltv']
X = df.drop(columns=drop_cols)
y_churn = df['target_churn']     # 모델 1 타겟
y_ltv = df['log_target_ltv']     # 모델 2 타겟 (로그 변환값)

# 8:2 비율로 데이터 분할 (이탈률 불균형을 유지하기 위해 stratify 사용)
X_train, X_val, y_churn_train, y_churn_val, y_ltv_train, y_ltv_val = train_test_split(
    X, y_churn, y_ltv, test_size=0.2, random_state=42, stratify=y_churn
)

print(f"📊 학습 데이터: {X_train.shape[0]}건 | 검증 데이터: {X_val.shape[0]}건\n")

📊 학습 데이터: 48000건 | 검증 데이터: 12000건



In [12]:
# ==========================================
# 3. [STAGE 1] 이탈률 예측 모델 (분류)
# ==========================================
print("▶️ [STAGE 1] 이탈 확률 예측 모델 학습 중...")
clf = lgb.LGBMClassifier(random_state=42, n_estimators=200, learning_rate=0.01, verbose=-1)
clf.fit(X_train, y_churn_train, eval_set=[(X_val, y_churn_val)])

# 검증셋 성능 확인 (ROC-AUC)
churn_val_pred = clf.predict_proba(X_val)[:, 1]
auc_score = roc_auc_score(y_churn_val, churn_val_pred)
print(f"✅ STAGE 1 완료! (검증셋 ROC-AUC Score: {auc_score:.4f})\n")

▶️ [STAGE 1] 이탈 확률 예측 모델 학습 중...
✅ STAGE 1 완료! (검증셋 ROC-AUC Score: 0.7766)



In [13]:
# ==========================================
# 4. [연결 고리] 예측된 이탈 확률을 새로운 Feature로 추가
# ==========================================
print("🔗 예측된 이탈 확률을 LTV 모델의 신규 Feature로 주입합니다...")
# 학습셋과 검증셋 각각에 이탈 예측 확률값을 새로운 컬럼으로 추가
X_train_ltv = X_train.copy()
X_val_ltv = X_val.copy()

X_train_ltv['pred_churn_prob'] = clf.predict_proba(X_train)[:, 1]
X_val_ltv['pred_churn_prob'] = churn_val_pred

🔗 예측된 이탈 확률을 LTV 모델의 신규 Feature로 주입합니다...


In [14]:
# ==========================================
# 5. [STAGE 2] LTV 예측 모델 (회귀)
# ==========================================
print("▶️ [STAGE 2] LTV 예측 모델 학습 중 (pred_churn_prob 포함)...")
# 회귀 모델은 새로운 피처(이탈 확률)가 포함된 X_train_ltv로 학습합니다.
reg = lgb.LGBMRegressor(random_state=42, n_estimators=300, learning_rate=0.01, verbose=-1)
reg.fit(X_train_ltv, y_ltv_train, eval_set=[(X_val_ltv, y_ltv_val)])

# 검증셋 예측 및 성능 평가
ltv_val_pred_log = reg.predict(X_val_ltv)

# 로그 변환했던 값을 다시 원래 금액(원) 단위로 복구 (expm1)
y_ltv_val_real = np.expm1(y_ltv_val)
ltv_val_pred_real = np.expm1(ltv_val_pred_log)

# 원래 금액 단위의 RMSE 계산
rmse = root_mean_squared_error(y_ltv_val_real, ltv_val_pred_real)
print(f"✅ STAGE 2 완료! (검증셋 RMSE: {rmse:,.0f}원)\n")

▶️ [STAGE 2] LTV 예측 모델 학습 중 (pred_churn_prob 포함)...
✅ STAGE 2 완료! (검증셋 RMSE: 1,513,041원)



In [ ]:
# LTV 모델 변수 중요도 상위 5개 확인
feature_imp = pd.DataFrame({'Feature': X_train_ltv.columns, 'Value': reg.feature_importances_})
top_features = feature_imp.sort_values(by='Value', ascending=False).head(10)
print("🏆 LTV 예측 모델 변수 중요도 Top 10:")
print(top_features.to_string(index=False))

🏆 LTV 예측 모델 변수 중요도 Top 10:
              Feature  Value
      pred_churn_prob    959
fin_asset_trend_score    681
         credit_score    626
          tenure_days    602
  card_spending_ratio    499
        recency_ratio    490
       total_spending    456
   spending_to_income    444
                  age    370
     income_x_deposit    357


In [15]:
final_score = 0.5 * auc_score + 0.5 * (1 / (1 + np.log(rmse)))

print("-" * 50)
print(f"✅ 검증 AUC: {auc_score:.4f}")
print(f"✅ 검증 RMSE: {rmse:,.0f} 원")
print(f"🏆 예상 최종 통합 점수: {final_score:.4f}")
print("-" * 50)

--------------------------------------------------
✅ 검증 AUC: 0.7766
✅ 검증 RMSE: 1,513,041 원
🏆 예상 최종 통합 점수: 0.4212
--------------------------------------------------
